In [39]:
import pandas as pd
from datetime import datetime
import numpy as np
from sqlalchemy import create_engine
import pymysql
from urllib.parse import quote_plus

# Configuration 

In [40]:
CSV_FILE = "C:\projects\etl_csv_to_mysql\data\sales.csv"

MYSQL_USER = "root"
MYSQL_PASSWORD = quote_plus("**************")
MYSQL_HOST = "localhost"    
MYSQL_PORT = "3306"
MYSQL_DATABASE = "sales_db"
TARGET_TABLE = "etl_transformed_sales"


## DataBase Connection 

In [41]:
# SQLALchemy Connection String 

conn_str = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

In [42]:
# Create SQLALchemy Engine 

engine = create_engine(conn_str)


# Read CSV (Extract) 

In [43]:
try:
    df = pd.read_csv(CSV_FILE)
    print("Data Extracted from {CSV_FILE} , shape = {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Error loading CSV",{e})
    raise


Data Extracted from {CSV_FILE} , shape = {df.shape}


,OrderId,Product,Category,SalesAmount,OrderDate,Region,CustomerName
0,1001,Laptop Pro 13,Electronics,1000.0,2025-01-05,West,Alice Johnson
1,1002,Mechanical Keyboard,Electronics,95.5,2025-01-06,East,Bob Smith
2,1003,Desk Chair Ergonomic,Furniture,350.0,2025-01-06,West,Charlie Brown
3,1004,Coffee Maker Deluxe,Home Goods,150.0,2025-01-07,Central,Diana Prince
4,1005,Wireless Mouse,Electronics,25.0,2025-01-07,East,Evan Peters


# Transform 

In [44]:
# Clean column names
df.columns = (
    df.columns
      .str.replace(' ', '_')
      .str.replace(r'([A-Z])', r'_\1', regex=True)
      .str.lower()
      .str.strip('_')
)

In [45]:
df

,order_id,product,category,sales_amount,order_date,region,customer_name
0,1001,Laptop Pro 13,Electronics,1000.00,2025-01-05,West,Alice Johnson
1,1002,Mechanical Keyboard,Electronics,95.50,2025-01-06,East,Bob Smith
2,1003,Desk Chair Ergonomic,Furniture,350.00,2025-01-06,West,Charlie Brown
3,1004,Coffee Maker Deluxe,Home Goods,150.00,2025-01-07,Central,Diana Prince
4,1005,Wireless Mouse,Electronics,25.00,2025-01-07,East,Evan Peters
5,1006,Monitor 27-inch 4K,Electronics,550.00,2025-01-08,West,Fiona Glenanne
6,1007,LED Desk Lamp,Home Goods,45.99,2025-01-08,South,George Costanza
7,1008,External SSD 1TB,Electronics,89.99,2025-01-09,Central,Hannah Montana
8,1009,Wood Bookshelf Small,Furniture,120.00,2025-01-09,South,Ian McKellen
9,1010,Gaming Headset,Electronics,-125.00,2025-01-10,East,Jasmine Teo


In [46]:
df["order_date"] = pd.to_datetime(df["order_date"])

In [47]:
df["sales_amount"] = df["order_date"]

In [48]:
# numeric conversion
df["sales_amount"] = pd.to_numeric(df["sales_amount"])

In [49]:
# Derived field
df["unit_price"] = df["sales_amount"]

In [50]:
# Categorical tier
conditions = [
    df["sales_amount"] >= 500,
    df["sales_amount"] >= 100
]
choices = ["High Value", "Medium Value"]

df["sales_tier"] = np.select(conditions, choices, default="Low Value")

In [51]:
df["sales_tier"]

0    High Value
1    High Value
2    High Value
3    High Value
4    High Value
5    High Value
6    High Value
7    High Value
8    High Value
9    High Value
Name: sales_tier, dtype: object

In [52]:
# Filter out non-positive values
df = df[df["sales_amount"] > 0]

In [53]:
# Add load timestamp
df["load_timestamp"] = pd.to_datetime(datetime.utcnow())

C:\Users\aryan\AppData\Local\Temp\ipykernel_13656\3865706313.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["load_timestamp"] = pd.to_datetime(datetime.utcnow())


In [54]:

print("Transformations complete.")
display(df.head())

Transformations complete.


,order_id,product,category,sales_amount,order_date,region,customer_name,unit_price,sales_tier,load_timestamp
0,1001,Laptop Pro 13,Electronics,1736035200000000000,2025-01-05,West,Alice Johnson,1736035200000000000,High Value,2026-06-04 12:17:24.638302
1,1002,Mechanical Keyboard,Electronics,1736121600000000000,2025-01-06,East,Bob Smith,1736121600000000000,High Value,2026-06-04 12:17:24.638302
2,1003,Desk Chair Ergonomic,Furniture,1736121600000000000,2025-01-06,West,Charlie Brown,1736121600000000000,High Value,2026-06-04 12:17:24.638302
3,1004,Coffee Maker Deluxe,Home Goods,1736208000000000000,2025-01-07,Central,Diana Prince,1736208000000000000,High Value,2026-06-04 12:17:24.638302
4,1005,Wireless Mouse,Electronics,1736208000000000000,2025-01-07,East,Evan Peters,1736208000000000000,High Value,2026-06-04 12:17:24.638302


In [55]:
df

,order_id,product,category,sales_amount,order_date,region,customer_name,unit_price,sales_tier,load_timestamp
0,1001,Laptop Pro 13,Electronics,1736035200000000000,2025-01-05,West,Alice Johnson,1736035200000000000,High Value,2026-06-04 12:17:24.638302
1,1002,Mechanical Keyboard,Electronics,1736121600000000000,2025-01-06,East,Bob Smith,1736121600000000000,High Value,2026-06-04 12:17:24.638302
2,1003,Desk Chair Ergonomic,Furniture,1736121600000000000,2025-01-06,West,Charlie Brown,1736121600000000000,High Value,2026-06-04 12:17:24.638302
3,1004,Coffee Maker Deluxe,Home Goods,1736208000000000000,2025-01-07,Central,Diana Prince,1736208000000000000,High Value,2026-06-04 12:17:24.638302
4,1005,Wireless Mouse,Electronics,1736208000000000000,2025-01-07,East,Evan Peters,1736208000000000000,High Value,2026-06-04 12:17:24.638302
5,1006,Monitor 27-inch 4K,Electronics,1736294400000000000,2025-01-08,West,Fiona Glenanne,1736294400000000000,High Value,2026-06-04 12:17:24.638302
6,1007,LED Desk Lamp,Home Goods,1736294400000000000,2025-01-08,South,George Costanza,1736294400000000000,High Value,2026-06-04 12:17:24.638302
7,1008,External SSD 1TB,Electronics,1736380800000000000,2025-01-09,Central,Hannah Montana,1736380800000000000,High Value,2026-06-04 12:17:24.638302
8,1009,Wood Bookshelf Small,Furniture,1736380800000000000,2025-01-09,South,Ian McKellen,1736380800000000000,High Value,2026-06-04 12:17:24.638302
9,1010,Gaming Headset,Electronics,1736467200000000000,2025-01-10,East,Jasmine Teo,1736467200000000000,High Value,2026-06-04 12:17:24.638302


# STEP 3: LOAD INTO MYSQL (Load)

In [56]:
try:
    df.to_sql(
        name=TARGET_TABLE,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print(f"Data successfully loaded into MySQL table: {TARGET_TABLE}")

except Exception as e:
    print(f"Error loading to MySQL: {e}")

Data successfully loaded into MySQL table: etl_transformed_sales
